# Equivariant Graph Neural Network for binding-site prediction

Third approach to the challenge. Each protein is modelled as a graph: one
node per residue (Cα atom), edges from a k-nearest-neighbour graph in 3-D.
Node features encode only the chemistry of the residue (one-hot amino
acid plus 7 physicochemical descriptors); the geometric signal is left to
the EGNN itself.

Reference: Satorras, Hoogeboom & Welling, *E(n) Equivariant Graph Neural
Networks*, ICML 2021.

EGNN message-passing (E(n)-equivariant for coords `x`, invariant for
features `h`):

    m_ij = phi_e(h_i, h_j, ||x_i - x_j||^2)
    x_i  <- x_i + sum_j (x_i - x_j) * phi_x(m_ij) / (n_i + 1)
    h_i  <- h_i + phi_h(h_i, sum_j m_ij)

## Architecture used here

* 28-dim node features (21 AA one-hot + 7 chemistry flags)
* 96 hidden units, 5 EGNN layers
* k = 16 nearest neighbours per node
* per-node logit head (LayerNorm → MLP → 1)
* BCE-with-logits loss, `pos_weight=8` to compensate the ≈ 10 % positive
  rate

The model has ≈ 340 k parameters and trains on CPU.

## 0  Setup

In [1]:
import os, sys, zipfile, pickle, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import jaccard_score

ROOT = Path('.').resolve()
if ROOT.name == '03_egnn':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / '03_egnn' / 'src'))
print('working dir:', ROOT)

# Data now lives as plain directories (data/train, data/test); the loader in
# pdb_source.py also accepts train.zip / test.zip if you happen to have them.
TRAIN_SRC = ROOT / 'data' / 'train'
TEST_SRC  = ROOT / 'data' / 'test'
TRAIN_CSV = ROOT / 'data' / 'train.csv'
MODEL_DIR = ROOT / '03_egnn' / 'model'; MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR   = ROOT / '03_egnn'

from data import parse_pdb_ca, NODE_FEAT_DIM
from egnn import EGNNBindingClassifier, knn_edges
from pdb_source import read_pdb_bytes
print('node feature dim:', NODE_FEAT_DIM)

working dir: /Users/amrabdullah/AI_2026ss/structural_bioinformatics/challenge
node feature dim: 28


## 1  Inspect features and edges on one protein

In [2]:
coords, feats, ids = parse_pdb_ca(read_pdb_bytes(str(TRAIN_SRC), 0))
print('residues       :', len(ids))
print('coords shape   :', coords.shape)
print('feats shape    :', feats.shape)
print('first residue  :', ids[0], 'features:', feats[0])

src, dst = knn_edges(torch.from_numpy(coords), k=16)
print('edges (k=16)   :', src.shape[0], '(', src.shape[0] / coords.shape[0], 'per node )')

residues       : 501
coords shape   : (501, 3)
feats shape    : (501, 28)
first residue  : A_5 features: [ 0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
  0.     0.     0.     0.     1.     0.     0.     0.     0.     0.
  0.    -0.9    0.867  1.     1.     0.     1.     0.   ]
edges (k=16)   : 8016 ( 16.0 per node )


## 2  Model

In [3]:
model = EGNNBindingClassifier(in_dim=NODE_FEAT_DIM, hidden=96, n_layers=5, k=16)
print('parameters:', sum(p.numel() for p in model.parameters()))

x = torch.from_numpy(coords); f = torch.from_numpy(feats)
with torch.no_grad():
    out = model(f, x)
print('forward output shape:', out.shape, '(should be (N,))')

parameters: 338795
forward output shape: torch.Size([501]) (should be (N,))


## 3  Training

Training is shell-driven (`approach3/code/train_egnn.py`) so it can run as
a long background job; we just invoke it from the notebook and read the
checkpoint back.

In [4]:
import subprocess

MODEL_PKL = MODEL_DIR / 'egnn_model.pkl'
if not MODEL_PKL.exists():
    cmd = [
        sys.executable, '03_egnn/src/train_egnn.py',
        '--archive', str(TRAIN_SRC),
        '--labels',  str(TRAIN_CSV),
        '--out',     str(MODEL_PKL),
        '--n_train', '3000', '--n_val', '300', '--epochs', '6',
        '--hidden', '96', '--n_layers', '5', '--k', '16',
        '--lr', '1e-3', '--pos_weight', '8.0',
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print(f'reusing existing checkpoint at {MODEL_PKL}')

payload = pickle.load(open(MODEL_PKL, 'rb'))
print('best val Jaccard:', payload['val_score'], 'threshold:', payload['threshold'])

reusing existing checkpoint at /Users/amrabdullah/AI_2026ss/structural_bioinformatics/challenge/03_egnn/model/egnn_model.pkl
best val Jaccard: 0.1719785575048733 threshold: 0.6599999999999999


## 4  Reload and verify on a held-out batch

In [5]:
model = EGNNBindingClassifier(
    in_dim=payload.get('in_dim', NODE_FEAT_DIM),
    hidden=payload['hidden'], n_layers=payload['n_layers'], k=payload['k'])
model.load_state_dict(payload['state_dict'])
model.eval()
threshold = payload['threshold']

df_labels = pd.read_csv(TRAIN_CSV)
label_map = {int(r['id']): set(str(r['resid']).split()) for _, r in df_labels.iterrows()}

rng = np.random.default_rng(20250618)
all_pids = np.array(sorted(label_map))
rng.shuffle(all_pids)
val_pids = all_pids[:300]

y_true, y_pred = [], []
with torch.no_grad():
    for pid in val_pids:
        data = read_pdb_bytes(str(TRAIN_SRC), pid)
        if data is None: continue
        coords, feats, ids = parse_pdb_ca(data)
        if not ids: continue
        x = torch.from_numpy(coords); f = torch.from_numpy(feats)
        prob = torch.sigmoid(model(f, x)).cpu().numpy()
        truth = label_map.get(int(pid), set())
        for r, q in zip(ids, prob):
            y_true.append(1 if r in truth else 0)
            y_pred.append(1 if q >= threshold else 0)
print(f'val flat Jaccard: {jaccard_score(y_true, y_pred):.4f}')

val flat Jaccard: 0.1720


## 5  Predict on test set

In [6]:
SUBMISSION = OUT_DIR / 'submission.csv'
cmd = [sys.executable, '03_egnn/src/predict_egnn.py',
       '--archive', str(TEST_SRC),
       '--model',   str(MODEL_PKL),
       '--out',     str(SUBMISSION)]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

df = pd.read_csv(SUBMISSION)
empty = (df['prediction'].fillna('').str.strip() == '').sum()
tokens = sum(len(str(p).split()) for p in df['prediction'].fillna(''))
print(f'rows={len(df)}  empty={empty}  total tokens={tokens}  avg/protein={tokens/(len(df)-empty):.1f}')
df.head(3)

/opt/anaconda3/envs/mlpc2026/bin/python 03_egnn/src/predict_egnn.py --archive /Users/amrabdullah/AI_2026ss/structural_bioinformatics/challenge/data/test --model /Users/amrabdullah/AI_2026ss/structural_bioinformatics/challenge/03_egnn/model/egnn_model.pkl --out /Users/amrabdullah/AI_2026ss/structural_bioinformatics/challenge/03_egnn/submission.csv


threshold = 0.660
scoring 3064 proteins from /Users/amrabdullah/AI_2026ss/structural_bioinformatics/challenge/data/test
  200/3064 (33.5/s)


  400/3064 (31.6/s)


  600/3064 (34.0/s)


  800/3064 (34.0/s)


  1000/3064 (34.5/s)


... (9 outputs left out)

  3000/3064 (34.7/s)


wrote /Users/amrabdullah/AI_2026ss/structural_bioinformatics/challenge/03_egnn/submission.csv in 88s


rows=3064  empty=0  total tokens=327678  avg/protein=106.9


,id,prediction
0,0,A_14 A_16 A_17 A_23 A_39 A_40 A_46 A_62 A_70 A...
1,1,A_2 A_3 A_4 A_5 A_6 A_16 A_17 A_19 A_21 A_33 A...
2,2,A_453 A_454 A_464 A_512 A_513 A_514 A_521 A_53...


## 6  Sanity check against `metric.py`

Build a fake target frame from `train.csv` for the same 300 held-out
proteins, then call the official scorer.

In [7]:
def parse_submission(df_submission):
    df_submission = df_submission.copy()
    df_submission['prediction'] = df_submission['prediction'].fillna('')
    rows = []
    for _, r in df_submission.iterrows():
        for res in str(r['prediction']).strip().split(' '):
            if res.strip():
                rows.append((r['id'], res.strip()))
    out = pd.DataFrame(rows, columns=['id', 'residue_id'])
    out['prediction'] = 1
    return out

target_rows = []
sub_rows = []
with torch.no_grad():
    for pid in val_pids:
        data = read_pdb_bytes(str(TRAIN_SRC), pid)
        if data is None: continue
        coords, feats, ids = parse_pdb_ca(data)
        truth = label_map.get(int(pid), set())
        for r in ids:
            target_rows.append({'id': pid, 'residue_id': r,
                                'true': 1 if r in truth else 0})
        x = torch.from_numpy(coords); f = torch.from_numpy(feats)
        prob = torch.sigmoid(model(f, x)).cpu().numpy()
        sel = [r for r, q in zip(ids, prob) if q >= threshold]
        sub_rows.append({'id': pid, 'prediction': ' '.join(sel)})
tgt = pd.DataFrame(target_rows)
sub = parse_submission(pd.DataFrame(sub_rows))
merged = pd.merge(tgt, sub, on=['id', 'residue_id'], how='left')
merged['prediction'] = merged['prediction'].fillna(0).astype(int)
print(f'metric.py replication: Jaccard = {jaccard_score(merged.true, merged.prediction):.4f}')

metric.py replication: Jaccard = 0.1720


## Summary

* 28-dim chemistry-only node features + Cα coordinates
* 5-layer EGNN, 96 hidden units, ≈ 340 k parameters
* trained on 3 000 proteins, evaluated on 300 held-out proteins
* model + threshold serialised to `03_egnn/model/egnn_model.pkl`
* test predictions written to `03_egnn/submission.csv`

Compared to the other approaches in this challenge:

| approach | model                              | local Jaccard |
| -------- | ---------------------------------- | ------------- |
| 1        | HistGradientBoosting (66 hand-crafted features) | 0.37 |
| 2        | P2Rank top-2 (pre-trained RF)      | 0.28          |
| **3**    | **EGNN (chemistry only)**          | ~0.17         |

The EGNN is the most parameter-efficient and novel, but under-trained at this
compute budget (only 3 000 of 16 379 proteins, 6 CPU epochs; validation
Jaccard was still climbing at cutoff). It is kept as a documented reference
approach — the graded submission is Approach 1.